In [1]:
import optuna
from optuna.samplers import TPESampler
from optuna.trial import Trial
from optuna.study import create_study
import diff_cont
import lambda_cont
import libs.agent_infra as ai
import os
import json
import datetime
import itertools
import constants as Cs
import concurrent.futures
import numpy as np
from concurrent.futures import ProcessPoolExecutor
SEEDS = [101, 102, 103]
TEST_EVAL_EPS = 6

/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-18 00:56:55.814876: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-18 00:56:55.848250: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-18 00:56:56.562547: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF

# Lambda

## Fitness

In [ ]:




def objective(trial:Trial):
    env = Cs.ENIVROMENTS["cartpole"]()
    cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
    l = trial.suggest_categorical("lambda",[10,20,30])
    m = trial.suggest_categorical("mu",[10,20,30])
    mr = trial.suggest_float("mutation_rate", 0, 0.1, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    sigma = trial.suggest_float("sigma", 0.5, 2, step=0.5)
    cross_uni = None
    if cross_method == "uniform":
        cross_uni = trial.suggest_float("cross", 0.1, 0.9, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.lambda_cont.argumented_function, env="cartpole",
                container="fitness",
                cross_method=cross_method,
                ng = 15,
                l = l,
                m = m,
                cr = cr,
                mr = mr,
                mutation_sigma = sigma,
                cross_uni = cross_uni if cross_uni is not None else 0.5,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
lambda_study = create_study(direction="maximize", sampler=sampler, storage="sqlite:///Data/optuna/cartpole/lambda.db", load_if_exists=True)
lambda_study.optimize(objective, n_trials=100, n_jobs=5)
print(lambda_study.best_trials)

[I 2026-05-18 00:24:32,674] A new study created in RDB with name: no-name-833753cb-dbca-411f-9984-503ede210e3d


gen	nevals	avg    	min	max    	std    
0  	10    	18.1333	9  	98.3333	26.7337
gen	nevals	avg    	min	max    	std    
0  	10    	18.9667	8  	41.3333	12.1193
gen	nevals	avg    	min	max    	std    
0  	10    	18.9667	8  	41.3333	12.1193
gen	nevals	avg    	min	max    	std    
0  	10    	18.1333	9  	98.3333	26.7337
gen	nevals	avg  	min	max    	std   
0  	20    	16.15	9  	98.3333	20.184
1  	8     	39.2   	9.66667	84.3333	25.2799
gen	nevals	avg    	min	max	std    
0  	20    	23.0833	8  	64 	18.3457
1  	10    	38.9667	9.33333	98.3333	33.2377
1  	9     	74.1333	9.33333	103    	38.5781
1  	13    	39.6   	10.3333	72.6667	18.3193
gen	nevals	avg    	min	max    	std   
0  	30    	21.4667	8  	81.6667	19.666
2  	10    	54.9333	15.3333	84.3333	24.0388
gen	nevals	avg    	min	max    	std   
0  	30    	21.4667	8  	81.6667	19.666
1  	8     	42.8 	9.33333	130.333	41.7084
gen	nevals	avg 	min    	max	std    
0  	10    	89.1	8.66667	500	141.467
gen	nevals	avg 	min    	max	std    
0  	10    	89.1	8.66667	500	14

[I 2026-05-18 00:26:57,478] Trial 3 finished with value: 373.1666666666667 and parameters: {'crossmethod': 'uniform', 'lambda': 10, 'mu': 10, 'mutation_rate': 0.0, 'cross_rate': 0.8, 'sigma': 0.5, 'cross': 0.30000000000000004}. Best is trial 3 with value: 373.1666666666667.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/cre

gen	nevals	avg    	min	max    	std    
0  	10    	18.1333	9  	98.3333	26.7337
gen	nevals	avg    	min	max    	std    
0  	10    	18.9667	8  	41.3333	12.1193
1  	12    	14.1   	9.33333	44.3333	10.6875
1  	11    	73.4   	20.3333	384.333	104.279
gen	nevals	avg 	min    	max	std    
0  	10    	89.1	8.66667	500	141.467
2  	12    	29.5667	9.33333	141.333	39.6101
1  	13    	111.867	10     	500	133.843


[I 2026-05-18 00:27:03,323] Trial 2 finished with value: 175.66666666666666 and parameters: {'crossmethod': 'mean', 'lambda': 10, 'mu': 20, 'mutation_rate': 0.03, 'cross_rate': 0.7, 'sigma': 1.5}. Best is trial 3 with value: 373.1666666666667.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A 

...

gen	nevals	avg    	min	max    	std    
0  	10    	18.1333	9  	98.3333	26.7337
gen	nevals	avg    	min	max    	std    
0  	10    	18.9667	8  	41.3333	12.1193


/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/

gen	nevals	avg    	min	max    	std    
0  	10    	18.9667	8  	41.3333	12.1193
gen	nevals	avg    	min	max    	std    
0  	10    	18.1333	9  	98.3333	26.7337
1  	11    	30.2333	9.66667	41.3333	11.1605
gen	nevals	avg 	min    	max	std    
0  	10    	89.1	8.66667	500	141.467
1  	12    	73.5667	9.33333	98.3333	35.7367
gen	nevals	avg 	min    	max	std    
0  	10    	89.1	8.66667	500	141.467
2  	19    	59.6333	34     	108    	24.30751  	16    	85.0333	10.3333	255.333	92.6199

1  	14    	211.667	56.3333	500	189.417
1  	16    	92.7   	9.33333	465    	128.752
1  	16    	285.933	56.3333	500	214.445
2  	11    	131.467	29.3333	267.333	92.2793
2  	7     	458.933	89.3333	500	123.2  
2  	14    	104.867	17.6667	255.333	85.7765
3  	16    	76.6   	41.3333	205.667	46.893 
3  	11    	500    	500    	500	0      
3  	10    	188.333	39     	267.333	87.3922
2  	16    	404.433	10     	500	191.211
2  	13    	190.5  	82.6667	465    	144.014
3  	18    	132.467	38.6667	255.333	70.7335


[I 2026-05-18 00:55:06,645] Trial 31 finished with value: 225.83333333333334 and parameters: {'crossmethod': 'mean', 'lambda': 30, 'mu': 20, 'mutation_rate': 0.02, 'cross_rate': 0.4, 'sigma': 0.5}. Best is trial 0 with value: 500.0.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named

4  	12    	254.467	203    	267.333	25.7333
4  	12    	450.933	9.33333	500	147.2  
4  	12    	113.033	11.6667	403.667	101.905
gen	nevals	avg    	min	max    	std    
0  	10    	18.9667	8  	41.3333	12.1193
gen	nevals	avg    	min	max    	std    
0  	10    	18.1333	9  	98.3333	26.7337
4  	17    	123    	16.6667	255.333	60.3466
5  	13    	114.533	17.6667	403.667	102.849
3  	14    	357.767	9      	500	217.363
3  	14    	240.133	98.3333	465    	155.439
5  	9     	500    	500    	500	0      
5  	15    	122.867	46.6667	131.333	25.4   
5  	15    	247.267	29     	305    	73.6171
gen	nevals	avg 	min    	max	std    
0  	10    	89.1	8.66667	500	141.467
1  	16    	85.0333	10.3333	255.333	92.6199
6  	11    	500    	500    	500	0      
4  	13    	272.2  	98.3333	465    	148.541
6  	16    	102.933	17.6667	173    	49.5111
4  	14    	421.367	61     	500	158.597
6  	16    	131.333	131.333	131.333	2.84217e-14
1  	16    	92.7   	9.33333	465    	128.752
1  	16    	285.933	56.3333	500	214.445
2  	14    	104.867

## Fit_Archive

In [4]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["cartpole"]()
    cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
    l = trial.suggest_categorical("lambda",[10,20,30])
    m = trial.suggest_categorical("mu",[10,20,30])
    mr = trial.suggest_float("mutation_rate", 0, 0.1, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    sigma = trial.suggest_float("sigma", 0.5, 2, step=0.5)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)

    cross_uni = None
    if cross_method == "uniform":
        cross_uni = trial.suggest_float("cross", 0.1, 0.9, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.lambda_cont.argumented_function, env="cartpole",
                container="fit_archiving",
                cross_method=cross_method,
                ng = 15,
                l = l,
                m = m,
                cr = cr,
                mr = mr,
                mutation_sigma = sigma,
                cross_uni = cross_uni if cross_uni is not None else 0.5,
                archiving_period=archiving,
                archive_batch=archive_batch,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
lambda_archive_study = create_study(direction="maximize", sampler=sampler, storage="sqlite:///Data/optuna/cartpole/fit_archiving/lambda.db", load_if_exists=True)
lambda_archive_study.optimize(objective, n_trials=200, n_jobs=5)

[I 2026-05-18 00:57:41,311] A new study created in RDB with name: no-name-0fb4f770-2e30-4fb5-8e62-bcbc0aef90a8


gen	nevals	avg    	min	max    	std    
0  	10    	18.9667	8  	41.3333	12.1193
gen	nevals	avg    	min	max    	std    
0  	10    	18.1333	9  	98.3333	26.7337
gen	nevals	avg    	min	max    	std    
0  	10    	18.1333	9  	98.3333	26.7337
gen	nevals	avg    	min	max    	std    
0  	10    	18.9667	8  	41.3333	12.1193
gen	nevals	avg  	min	max    	std   
0  	20    	16.15	9  	98.3333	20.184
1  	7     	48.3   	9  	98.3333	42.0192
gen	nevals	avg  	min	max    	std   
0  	20    	16.15	9  	98.3333	20.184
gen	nevals	avg    	min	max	std    
0  	20    	23.0833	8  	64 	18.3457
gen	nevals	avg    	min	max	std    
0  	20    	23.0833	8  	64 	18.3457
1  	7     	30.1667	8.66667	54.3333	17.23  
gen	nevals	avg    	min	max    	std   
0  	30    	21.4667	8  	81.6667	19.666
1  	8     	39.2167	9.33333	111.667	37.9819
1  	10    	44.1833	8.66667	120.333	30.9793
1  	7     	58.55  	9.66667	120.333	30.035 
gen	nevals	avg 	min    	max	std    
0  	10    	89.1	8.66667	500	141.467
1  	10    	43.5167	9.33333	264    	76.661
gen

[W 2026-05-18 00:58:22,822] Trial 4 failed with parameters: {'crossmethod': 'uniform', 'lambda': 10, 'mu': 30, 'mutation_rate': 0.02, 'cross_rate': 0.6000000000000001, 'sigma': 2.0, 'archiving_period': 3, 'archive_batch': 5, 'cross': 0.6} because of the following error: KeyboardInterrupt().
concurrent.futures.process._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/concurrent/futures/process.py", line 246, in _process_worker
    r = call_item.fn(*call_item.args, **call_item.kwargs)
  File "/home/schkliba/git/MastersThesis/lambda_cont.py", line 127, in argumented_function
    alg.run()
  File "/home/schkliba/git/MastersThesis/containers.py", line 96, in run
    self.final_pop, self.logbook = es.eaMuPlusLambda(
  File "/home/schkliba/git/MastersThesis/evolutionary_strategy.py", line 102, in eaMuPlusLambda
    for ind, fit in zip(invalid_ind, fitnesses):
  File "/home/schkliba/git/MastersThesis/libs/agent_infra.py", li

KeyboardInterrupt: 

## Novelty

In [2]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["cartpole"]()
    cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
    l = trial.suggest_categorical("lambda",[10,20,30])
    m = trial.suggest_categorical("mu",[10,20,30])
    mr = trial.suggest_float("mutation_rate", 0, 0.1, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    sigma = trial.suggest_float("sigma", 0.5, 2, step=0.5)

    cross_uni = None
    if cross_method == "uniform":
        cross_uni = trial.suggest_float("cross", 0.1, 0.9, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.lambda_cont.argumented_function, env="cartpole",
                container="novelty",
                cross_method=cross_method,
                ng = 15,
                l = l,
                m = m,
                cr = cr,
                mr = mr,
                mutation_sigma = sigma,
                cross_uni = cross_uni if cross_uni is not None else 0.5,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
lambda_novelty_study = create_study(direction="maximize", sampler=sampler, storage="sqlite:///Data/optuna/cartpole/novelty/lambda.db", load_if_exists=True)
lambda_novelty_study.optimize(objective, n_trials=150, n_jobs=5)

[I 2026-05-18 00:57:02,618] A new study created in RDB with name: no-name-a5ca42d6-82d1-4f7e-beb7-778ef666cd7d


   	      	                    fitness                    	                            novelty                             
   	      	-----------------------------------------------	----------------------------------------------------------------
gen	nevals	avg    	gen	max    	min	nevals	std    	avg     	gen	max    	min     	nevals	std     
0  	10    	18.1333	0  	98.3333	9  	10    	26.7337	0.361959	0  	0.93749	0.224966	10    	0.274262
   	      	                    fitness                    	                                novelty                                 
   	      	-----------------------------------------------	------------------------------------------------------------------------
gen	nevals	avg    	gen	max    	min	nevals	std    	avg     	gen	max     	min     	nevals	std     
0  	10    	18.9667	0  	41.3333	8  	10    	12.1193	0.485193	0  	0.824636	0.346479	10    	0.207662
   	      	                   fitness                    	                            novelty         

[W 2026-05-18 00:57:12,719] Trial 1 failed with parameters: {'crossmethod': 'uniform', 'lambda': 30, 'mu': 10, 'mutation_rate': 0.05, 'cross_rate': 0.7, 'sigma': 2.0, 'cross': 0.5} because of the following error: KeyboardInterrupt().
concurrent.futures.process._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/concurrent/futures/process.py", line 246, in _process_worker
    r = call_item.fn(*call_item.args, **call_item.kwargs)
  File "/home/schkliba/git/MastersThesis/lambda_cont.py", line 127, in argumented_function
    alg.run()
  File "/home/schkliba/git/MastersThesis/containers.py", line 96, in run
    self.final_pop, self.logbook = es.eaMuPlusLambda(
  File "/home/schkliba/git/MastersThesis/evolutionary_strategy.py", line 82, in eaMuPlusLambda
    fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
  File "/home/schkliba/git/MastersThesis/containers.py", line 44, in novelty_operator
    behaviours = list(eval_m

KeyboardInterrupt: 

# Diff

## Fitness

In [2]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["cartpole"]()
    l = trial.suggest_categorical("pop",[5,10,15,20])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, 
                env="cartpole",
                container="fitness",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
study = create_study(direction="maximize", sampler=sampler, storage="sqlite:///Data/optuna/cartpole/diff.db", load_if_exists=True)
study.optimize(objective, n_trials=100, n_jobs=5)
print(study.best_trials)

[I 2026-05-16 17:56:12,308] A new study created in RDB with name: no-name-d6786c8a-3090-4943-a83e-3e361db38748


gen	avg    	min	max    	std    
0  	33.5333	9  	106.667	37.7404
gen	avg	min	max    	std   
0  	43 	9  	111.667	38.406
1  	49 	9.33333	111.667	37.901
gen	avg    	min    	max	std    
0  	91.1333	9.66667	235	77.6942
1  	70.8667	9.33333	218.333	77.9032
1  	101.467	42.6667	235	69.1365
gen	avg    	min    	max    	std    
0  	57.0667	9.66667	175.333	55.6942gen	avg    	min    	max    	std    
0  	62.5333	9.33333	234.333	64.1525

2  	78.8	9.33333	174.333	55.1098
2  	101.467	42.6667	235	69.1365
gen	avg    	min	max    	std   
0  	80.0333	24 	323.333	87.852
3  	110.933	42.6667	235	67.3985
2  	139.4  	9.66667	336.667	120.815
gen	avg    	min    	max	std    
0  	102.933	9.33333	500	136.274
gen	avg    	min    	max    	std    
0  	47.3833	8.66667	144.667	33.6109
1  	88.9   	38.6667	323.333	84.0603
4  	113.2  	54     	235	65.2198
3  	169.733	55.6667	466.333	154.467
5  	113.2  	54     	235	65.2198
1  	113.4  	10.3333	500    	138.689
4  	170.267	55.6667	466.333	154.079
6  	113.2  	54     	235	65.2198
gen	

[I 2026-05-16 17:57:38,571] Trial 3 finished with value: 272.1666666666667 and parameters: {'pop': 5, 'mutation_rate': 0.8, 'cross_rate': 0.5}. Best is trial 3 with value: 272.1666666666667.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and

9  	354.9  	80     	500	145.698
14 	296.367	89     	500    	164.637
4  	209.833	28.3333	500	163.584
9  	293.133	69     	500	135.471
6  	176.683	10     	500	132.578
11 	281.567	120    	500	128.898
7  	201.383	34.3333	500    	163.269
8  	177.283	54.6667	500    	127.333
5  	228.833	10     	500    	145.058
12 	281.567	120    	500	128.898
10 	293.133	69     	500	135.471
10 	364.267	173.667	500	129.876
10 	402.933	135.333	500    	135.068
gen	avg    	min    	max	std    
0  	27.9556	9.33333	131	31.5498
5  	299.317	61.3333	500	173.606
gen	avg    	min    	max    	std    
0  	51.6222	8.66667	289.333	66.0244
11 	295.3  	90.6667	500	131.998
11 	368    	211    	500	124.78 
1  	64.3778	8.66667	289.333	65.327 
5  	224.483	28.3333	500	157.832
9  	187.917	62.3333	500    	123.492
12 	368    	211    	500	124.78 
8  	218.483	34.3333	500    	157.206
13 	311.5  	132.667	500	122.5  
1  	96.8889	9.33333	500	120.43 
11 	419.2  	195.333	500    	109.174
2  	81.2889	10.3333	289.333	65.9745
12 	314.333	90.6667	500	

[I 2026-05-16 17:59:16,967] Trial 2 finished with value: 447.1666666666667 and parameters: {'pop': 10, 'mutation_rate': 0.7000000000000001, 'cross_rate': 0.3}. Best is trial 2 with value: 447.1666666666667.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already 

...

14 	190.05 	61.3333	500    	126.968
8  	253.75 	52.3333	500    	140.526
9  	317.633	55.3333	500	165.002
9  	295.7  	69     	500    	141.305
10 	354.2  	68     	500	150.17 
10 	302.8  	69     	500    	147.681
11 	358.883	68     	500	153.154
11 	314.5  	69     	500    	153.458
12 	364.067	68     	500	146.936


[I 2026-05-16 19:08:18,324] Trial 98 finished with value: 411.0 and parameters: {'pop': 20, 'mutation_rate': 0.5, 'cross_rate': 0.3}. Best is trial 1 with value: 500.0.


12 	317.183	104.667	500    	149.505
13 	376.683	76     	500	144.873
13 	329.067	130.667	500    	136.761
14 	376.683	76     	500	144.873
14 	329.067	130.667	500    	136.761


[I 2026-05-16 19:09:09,017] Trial 99 finished with value: 500.0 and parameters: {'pop': 20, 'mutation_rate': 0.5, 'cross_rate': 0.3}. Best is trial 1 with value: 500.0.


[FrozenTrial(number=1, state=<TrialState.COMPLETE: 1>, values=[500.0], datetime_start=datetime.datetime(2026, 5, 16, 17, 56, 12, 320274), datetime_complete=datetime.datetime(2026, 5, 16, 17, 59, 37, 808243), params={'pop': 10, 'mutation_rate': 0.30000000000000004, 'cross_rate': 1.0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'pop': CategoricalDistribution(choices=(5, 10, 15, 20)), 'mutation_rate': FloatDistribution(high=1.0, log=False, low=0.0, step=0.1), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1)}, trial_id=22, value=None), FrozenTrial(number=4, state=<TrialState.COMPLETE: 1>, values=[500.0], datetime_start=datetime.datetime(2026, 5, 16, 17, 56, 12, 377013), datetime_complete=datetime.datetime(2026, 5, 16, 18, 2, 1, 945776), params={'pop': 20, 'mutation_rate': 0.30000000000000004, 'cross_rate': 0.4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'pop': CategoricalDistribution(choices=(5, 10, 15, 20)), '

## Fit Archiving

In [ ]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["cartpole"]()
    l = trial.suggest_categorical("pop",[5,10,15,20])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, 
                env="cartpole",
                container="fit_archiving",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                archiving_period=archiving,
                archive_batch=archive_batch,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
study = create_study(direction="maximize", sampler=sampler, storage="sqlite:///Data/optuna/cartpole/fit_archiving/diff.db", load_if_exists=True)
study.optimize(objective, n_trials=150, n_jobs=5)
print(study.best_trials)

[I 2026-05-18 00:59:15,089] A new study created in RDB with name: no-name-21cbce2a-4dfb-4e3b-aec4-c29e0ba9d7f6
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or re

gen	avg    	min	max    	std    
0  	10.1333	9  	13.6667	1.77138
gen	avg    	min	max    	std    
0  	43.2667	9  	111.667	38.3796
gen	avg 	min    	max    	std    
0  	52.6	9.66667	82.3333	25.3495
gen	avg    	min    	max    	std    
0  	50.5667	10.3333	162.667	39.9177
1  	89.1333	69.6667	123    	17.9946
gen	avg 	min	max	std   
0  	63.9	9  	349	99.624
1  	133.2  	56.3333	268    	71.0323
2  	91.6667	82.3333	123    	15.8381
gen	avg    	min    	max	std    
0  	86.9667	9.33333	500	141.085
gen	avg 	min    	max    	std    
0  	91.9	8.66667	493.333	138.252
gen	avg    	min    	max    	std    
0  	56.3333	8.66667	204.333	51.8662
3  	101    	82.3333	123    	18.096 
1  	192.733	14 	500    	173.892
2  	144.267	111.667	268    	61.9205
1  	129.8	35.6667	349	111.599
gen	avg  	min    	max    	std    
0  	44.45	8.66667	126.333	31.9225
gen	avg    	min	max	std    
0  	116.333	9  	500	133.519
1  	120.933	47.6667	500    	133.348
gen	avg    	min    	max	std    
0  	77.0444	9.33333	500	126.101
gen	avg    	min   

[I 2026-05-18 01:01:32,470] Trial 2 finished with value: 178.5 and parameters: {'pop': 5, 'mutation_rate': 0.6000000000000001, 'cross_rate': 0.9000000000000001, 'archiving_period': 2, 'archive_batch': 5}. Best is trial 2 with value: 178.5.


13 	500    	500    	500    	0      


/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/

...

/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/

gen	avg	min    	max	std    
0  	16 	9.33333	31 	8.75595
5  	455.8  	163    	500    	97.7791
gen	avg    	min    	max	std    
0  	50.8667	8.66667	90 	26.4731
14 	500    	500    	500	0      
gen	avg    	min	max	std    
0  	89.6667	9  	161	67.9879
1  	89.4667	59.6667	148	32.2577
3  	500    	500    	500	0      
7  	263.467	234.333	500    	79.0014
gen	avg    	min    	max    	std    
0  	62.6667	9.33333	237.333	63.4798
gen	avg 	min	max	std    
0  	64.1	27 	146	40.9865
2  	113.2  	90     	148	28.4141
3  	148    	148    	148	0      
1  	216.867	157.333	447.667	115.412
12 	500    	500    	500	0      
4  	148    	148    	148	0      
1  	92.2333	43.3333	146	41.1515
1  	273.6	81.3333	500	156.905
6  	489.5  	485    	500    	6.87386
8  	264.733	234.333	500    	78.6248
1  	140.833	34.6667	241.333	83.322 
5  	148    	148    	148	0      
gen	avg    	min    	max	std    
0  	165.333	9.66667	500	170.857
6  	148    	148    	148	0      
2  	295.267	161    	447.667	129.477
2  	114.8  	53     	146	35.8792
4  	

[I 2026-05-18 01:02:22,422] Trial 4 finished with value: 75.33333333333333 and parameters: {'pop': 15, 'mutation_rate': 0.1, 'cross_rate': 0.7, 'archiving_period': 5, 'archive_batch': 2}. Best is trial 1 with value: 500.0.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individu

2  	399.2  	134.333	500	155.623
5  	145.4  	134.333	151.667	4.05737
5  	500    	500    	500	0      
5  	447.667	447.667	447.667	0      
9  	326.233	247    	500    	113.355
3  	298.967	233.667	381    	63.3373
9  	239.467	174.333	500    	130.267
8  	500    	500    	500    	0      
6  	149.533	146    	175.667	8.87343
6  	447.667	447.667	447.667	0      
3  	500    	500    	500	0      
3  	472.967	229.667	500	81.1   
9  	500    	500    	500    	0      
14 	500    	500    	500	0      
6  	500    	500    	500	0      
12 	500    	500    	500	0      
7  	447.667	447.667	447.667	0      
7  	168.433	146    	299.667	45.1723
4  	325.8  	237.333	381    	59.2511
10 	304.6  	174.333	500    	159.543
8  	447.667	447.667	447.667	0      
10 	500    	500    	500    	0      
4  	500    	500    	500	0      
10 	387.033	247    	500    	114.908
11 	434.867	174.333	500    	130.267
8  	183.867	146    	299.667	51.1588
gen	avg  	min    	max	std    
0  	66.25	9.33333	500	115.441
gen	avg  	min    	max	std    
0  	82

Process ForkProcess-65:
Process ForkProcess-69:
Process ForkProcess-62:
Process ForkProcess-64:
Process ForkProcess-63:
Process ForkProcess-53:
Process ForkProcess-52:
Process ForkProcess-61:
Process ForkProcess-68:
Process ForkProcess-66:
Process ForkProcess-56:
Process ForkProcess-67:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback (most recent call last):
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback (most recent call last):
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multi

## Novelty

In [5]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["cartpole"]()
    l = trial.suggest_categorical("pop",[5,10,15,20,30])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)

    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, 
                env="cartpole",
                container="novelty",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
study = create_study(direction="maximize", sampler=sampler, storage="sqlite:///Data/optuna/cartpole/fit_archiving/diff.db", load_if_exists=True)
study.optimize(objective, n_trials=150, n_jobs=5)
print(study.best_trials)

[I 2026-05-18 00:58:29,586] A new study created in RDB with name: no-name-f4a4a8ed-c6a1-44d3-be1c-c4fc9dab4299


   	                fitness                	                            novelty                             
   	---------------------------------------	----------------------------------------------------------------
gen	avg 	gen	max	min	std    	avg     	gen	max     	min     	std     
0  	26.8	0  	80 	8  	27.5089	0.392555	0  	0.927338	0.252952	0.267472
   	                fitness                	                             novelty                             
   	---------------------------------------	-----------------------------------------------------------------
gen	avg    	gen	max    	min	std    	avg     	gen	max      	min      	std      
0  	10.3333	0  	15.6667	9  	2.66667	0.018021	0  	0.0450525	0.0112631	0.0135158
1  	19.6667	1  	53.6667	9  	17.3282	0.339485	1  	0.818571 	0.214239 	0.239693 
2  	17.0667	2  	48.6667	9  	15.8021	0.343202	2  	0.830563 	0.216284 	0.243807 
   	                fitness                	                            novelty                             

Process ForkProcess-32:
Process ForkProcess-31:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/concurrent/futures/process.py", line 240, in _process_worker
    call_item = call_queue.get(block=True)
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/queues.py", line 103, in get
    res = self._recv_bytes()
  File "/hom

KeyboardInterrupt: 

In [3]:
study.best_trials

[FrozenTrial(number=1, state=<TrialState.COMPLETE: 1>, values=[500.0], datetime_start=datetime.datetime(2026, 5, 16, 17, 56, 12, 320274), datetime_complete=datetime.datetime(2026, 5, 16, 17, 59, 37, 808243), params={'pop': 10, 'mutation_rate': 0.30000000000000004, 'cross_rate': 1.0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'pop': CategoricalDistribution(choices=(5, 10, 15, 20)), 'mutation_rate': FloatDistribution(high=1.0, log=False, low=0.0, step=0.1), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1)}, trial_id=22, value=None),
 FrozenTrial(number=4, state=<TrialState.COMPLETE: 1>, values=[500.0], datetime_start=datetime.datetime(2026, 5, 16, 17, 56, 12, 377013), datetime_complete=datetime.datetime(2026, 5, 16, 18, 2, 1, 945776), params={'pop': 20, 'mutation_rate': 0.30000000000000004, 'cross_rate': 0.4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'pop': CategoricalDistribution(choices=(5, 10, 15, 20)), 